# Annealed Gromov signatures — numerical and NYC TLC study

**Companion to the mathematical manuscript.** Contains all Python source; runs directly in Colab. Exact rational calculations, Monte Carlo size/power, computational benchmarks, deliberate model misspecification, official NYC TLC pickup-zone analysis, publication-oriented figures and tables, and a single downloadable ZIP.

**Important:** `smoke` and `pilot` outputs are preliminary; the TLC p-values are exploratory because stationarity and finite-range dependence are unverified. No TLC outputs are fabricated when downloads fail.

## 1 · Environment and reproducibility
The only external download in simulation-only mode is the Python dependency installer. Real-data mode additionally downloads **official** TLC monthly Parquet trip files and taxi-zone metadata. Prefer runtime with enough disk and RAM for streaming monthly Parquet files.

In [ ]:
%pip -q install "pyarrow>=14" "pyshp>=2.3" "shapely>=2" "pyproj>=3.5"

## 2 · Run configuration
Choose `smoke` first; use `pilot` for examination and `paper` for the high-budget study. `MODE='simulation'` avoids any TLC data download. `MODE='real'` runs only the real-data pipeline; `MODE='all'` does both. Paper-mode TLC covers January 2018 through December 2024 by default. You may provide `START_MONTH` and `END_MONTH` as `YYYY-MM` to override.

In [ ]:
PROFILE = "pilot"   # "smoke", "pilot", or "paper"
MODE = "all"       # "simulation", "real", or "all"
MASTER_SEED = 20261024
START_MONTH = ""    # e.g. "2020-01"; leave BOTH empty for profile defaults
END_MONTH = ""      # e.g. "2024-12"
KEEP_PARQUET = False  # True keeps large raw monthly Parquet data in Colab cache

assert PROFILE in {"smoke", "pilot", "paper"}
assert MODE in {"simulation", "real", "all"}
assert bool(START_MONTH) == bool(END_MONTH), "Specify both start and end months"
print("Protocol:", PROFILE, MODE, "seed", MASTER_SEED)

## 3 · Self-contained, editable Python implementation
The full engine is embedded below and written to `annealed_study.py` when this cell runs. Edit the module here to change experiments or add estimators; every run copies the executed source into its output ZIP.

In [ ]:
%%writefile annealed_study.py
#!/usr/bin/env python3
"""Annealed Gromov signatures: reproducible simulations and NYC TLC protocol.

Methodological scope: finite coordinate approximation with independent incomplete
U-statistics for synthetic observed clouds; Monte Carlo integration of the
empirical monthly probability measure for the TLC application.  Gaussian bridge
calibration uses the non-overlapping positive-semidefinite block estimator from
Eq. (blocks)-(bootstrap) of the companion theory manuscript. This is NOT an
implementation of the infinite characteristic feature embedding.

Example: python annealed_study.py --profile smoke --mode simulation
         python annealed_study.py --profile pilot --mode all
         python annealed_study.py --profile paper --mode all
The zip of generated outputs is created on normal exit, including partial runs
when the real-data download fails (with an explicit error receipt).
"""
from __future__ import annotations
import argparse
import hashlib
import importlib.metadata
import itertools
import json
import math
import os
import platform
import shutil
import sys
import time
import traceback
import urllib.request
import zipfile
from datetime import datetime, timezone
from fractions import Fraction
from math import comb
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.collections import PolyCollection
from scipy.special import ndtr

ROOT = Path(__file__).resolve().parent
COLORS = {"ink": "#13243C", "blue": "#2764A5", "teal": "#14877B", "orange": "#D27A37", "red": "#B64D58", "gray": "#7A8594"}


def visual_style():
    plt.rcParams.update({"figure.dpi":115, "savefig.dpi":240,
                         "font.family":"DejaVu Sans", "font.size":10.5,
                         "axes.titlesize":12, "axes.labelsize":10.5,
                         "axes.spines.top":False,"axes.spines.right":False,
                         "grid.alpha":0.18,"axes.grid":True,
                         "legend.frameon":False,"figure.facecolor":"white"})


def save_fig(fig, folder: Path, basename: str):
    folder.mkdir(parents=True, exist_ok=True)
    for suffix in ("png", "pdf"):
        fig.savefig(folder / f"{basename}.{suffix}", bbox_inches="tight", facecolor="white")
    plt.close(fig)


def write_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=lambda x: x.item() if isinstance(x, np.generic) else str(x))


def write_table(df: pd.DataFrame, folder: Path, name: str, caption: str = ""):
    folder.mkdir(parents=True, exist_ok=True)
    df.to_csv(folder/f"{name}.csv", index=False, float_format="%.10g")
    with open(folder/f"{name}.tex", "w", encoding="utf-8") as f:
        if caption: f.write(f"% {caption}\n")
        f.write(df.to_latex(index=False, escape=True, float_format=lambda x:f"{x:.4g}"))
    try:
        from IPython.display import display
        print(f"\n{name} — {caption}")
        display(df.head(18))
    except Exception:
        print(df.head(12).to_string(index=False))


def archive_results(base: Path):
    target = base.parent / f"annealed_gromov_outputs_{base.name}.zip"
    with zipfile.ZipFile(target, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for p in sorted(base.rglob("*")):
            if p.is_file() and p.resolve() != target.resolve():
                zf.write(p, arcname=str(p.relative_to(base.parent)))
    print("\nOUTPUT ZIP:", target, "size MB",round(target.stat().st_size/1e6,2))
    return target


# 1. Finite-space constructions and exact reference events -----------------

def graph_metric(name: str, scale=1., masses=None):
    edge_sets = {"star":((0,1),(0,2),(0,3)),
                 "path":((0,1),(1,2),(2,3)),
                 "complete":tuple((i,j) for i in range(4) for j in range(i+1,4)),
                 "cycle":((0,1),(1,2),(2,3),(3,0))}
    if name not in edge_sets: raise ValueError(name)
    d = np.full((4,4),2.0*scale)
    np.fill_diagonal(d,0.)
    for a,b in edge_sets[name]: d[a,b]=d[b,a]=scale
    p = np.ones(4)/4 if masses is None else np.asarray(masses,dtype=float)
    if not np.isclose(p.sum(),1.) or np.any(p<0): raise ValueError("invalid masses")
    return (d,p)


def exact_pattern_law(name, k):
    edges = {"star":{(0,1),(0,2),(0,3)},
             "path":{(0,1),(1,2),(2,3)}}[name]
    counts = {}
    for labels in itertools.product(range(4), repeat=k):
        pat = tuple(0 if labels[i] == labels[j] else
                    (1 if tuple(sorted((labels[i],labels[j]))) in edges else 2)
                    for i in range(k) for j in range(i+1,k))
        counts[pat] = counts.get(pat,0)+1
    return {pat:Fraction(v,4**k) for pat,v in counts.items()}


def finite_difference_mixture(M):
    ps=[Fraction(i+1,2*(M+3)) for i in range(M+2)]
    c=[(-1)**i*comb(M+1,i) for i in range(M+2)]
    a=[Fraction(max(0,v),2**M) for v in c]
    b=[Fraction(max(0,-v),2**M) for v in c]
    return ps,a,b


def two_atom_pattern(p,k):
    d={}
    for v in itertools.product(range(2),repeat=k):
        pat=tuple(int(v[i]!=v[j]) for i in range(k) for j in range(i+1,k))
        w=p**sum(vv==0 for vv in v)*(1-p)**sum(vv==1 for vv in v)
        d[pat]=d.get(pat,Fraction(0))+w
    return d


def mixed_two_atom(ps, weights, k):
    d={}
    for p,w in zip(ps,weights):
        if not w: continue
        for pat,mass in two_atom_pattern(p,k).items():
            d[pat]=d.get(pat,Fraction(0))+w*mass
    return d


def exact_verification(out: Path, max_M=8):
    ex=out/"01_exact"; ex.mkdir(parents=True,exist_ok=True)
    a=exact_pattern_law("star",2); b=exact_pattern_law("path",2)
    assert a==b=={(0,):Fraction(1,4),(1,):Fraction(3,8),(2,):Fraction(3,8)}
    triple=(1,1,2)
    assert exact_pattern_law("star",3)[triple]==Fraction(3,32)
    assert exact_pattern_law("path",3)[triple]==Fraction(1,16)
    rows=[]
    for M in [2,3,4,6,8]:
        if M>max_M: continue
        ps,wa,wb=finite_difference_mixture(M)
        for k in range(1,M+1):
            assert mixed_two_atom(ps,wa,k)==mixed_two_atom(ps,wb,k)
        sep=M+1 if (M+1)%2==0 else M+2
        # Exact witness: all sampled points have the same value.
        diff=sum((v-w)*(p**sep+(1-p)**sep) for p,v,w in zip(ps,wa,wb))
        assert diff!=0
        rows.append(dict(M=M, first_separating_order=sep,
                         exact_probability_difference=str(diff),
                         absolute_difference=float(abs(diff)),
                         verified_equal_through_M=True))
    write_table(pd.DataFrame(rows),ex,"moment_matching_exact",
                "Exact rational finite-cloud nonidentifiability: the first distinguishing order")
    triple_law_a=exact_pattern_law("star",3); triple_law_b=exact_pattern_law("path",3)
    patterns=sorted(set(triple_law_a)|set(triple_law_b))
    pats=pd.DataFrame([dict(pattern=str(x),star=float(triple_law_a.get(x,0)),
                           path=float(triple_law_b.get(x,0))) for x in patterns])
    pats["difference"]=pats.star-pats.path
    write_table(pats,ex,"triple_patterns","Labelled triple-distance event probabilities")
    fig,ax=plt.subplots(figsize=(9,4.3))
    idx=np.arange(len(pats)); top=pats.reindex(pats.difference.abs().sort_values(ascending=False).index).head(9)
    idx=np.arange(len(top))
    ax.bar(idx-.19,top.star,.38,color=COLORS["blue"],label="Star")
    ax.bar(idx+.19,top.path,.38,color=COLORS["orange"],label="Path")
    ax.set_xticks(idx,top.pattern,rotation=45,ha="right");ax.set_ylabel("Exact event probability")
    ax.set_title("Order-three geometry distinguishes the star from the path")
    ax.legend(loc="upper center",bbox_to_anchor=(.5,1.14),ncol=2)
    save_fig(fig,ex,"fig01_exact_triple_patterns")
    write_json(ex/"exact_assertions.json",{"pair_distribution":{str(k):str(v) for k,v in a.items()},
               "wedge_star":"3/32","wedge_path":"1/16","verified":True})


# 2. Finite feature dictionary, shared at all times -------------------------

def dictionary_for_metric(max_k=3,R=6,domain="graph"):
    # The dictionary is prespecified; its finite truncation is not itself omnibus.
    if domain=="graph":
        levels=[0.,1.,2.,.5,1.5]
        triple=[[1.,1.,2.],[1.,2.,1.],[2.,1.,1.],[1.,1.,1.],[2.,2.,2.],[0.,0.,0.]]
    elif domain=="nyc":
        levels=[0.,.25,.5,1.,1.5]
        triple=[[.25,.25,.5],[.25,.5,.25],[.5,.25,.25],[.5,.5,.5],[1.,1.,1.],[0.,0.,0.]]
    else: raise ValueError(domain)
    rng=np.random.default_rng(1871)  # fixed dictionary independent of all observations
    centers={}
    for k in range(2,max_k+1):
        d=k*(k-1)//2; vv=[]
        if k==2: vv=[[x] for x in levels]
        elif k==3: vv=triple.copy()
        else:
            vv=[np.zeros(d).tolist(),np.ones(d).tolist(),
                (np.ones(d)*(2. if domain=="graph" else .5)).tolist()]
        while len(vv)<R:
            # deterministic additional centres, NOT learned from current data
            vv.append(rng.choice(levels,size=d,replace=True).tolist())
        centers[k]=np.asarray(vv[:R],dtype=float)
    return centers


def weighted_features(distances: np.ndarray, centers, bandwidth=1.):
    # distances: (L, choose(k,2)); centres: (R, choose(k,2))
    # Gaussian e^{-||D-q||^2 / bandwidth^2}, exactly the companion feature form
    dif=distances[:,None,:]-centers[None,:,:]
    return np.exp(-np.einsum("lrd,lrd->lr",dif,dif,optimize=True)/(bandwidth**2)).mean(axis=0)


def sample_distinct_rows(rng,m,L,k):
    # Uniform ordered *distinct observation indices* via rejection sampling;
    # drawing i.i.d integers and rejecting collisions is exactly uniform.
    if k>m: raise ValueError("m must be at least k")
    out=np.empty((L,k),dtype=np.int32)
    left=np.arange(L)
    while left.size:
        candidate=rng.integers(0,m,size=(left.size,k),dtype=np.int32)
        sorted_c=np.sort(candidate,axis=1)
        good=np.all(np.diff(sorted_c,axis=1)!=0,axis=1)
        out[left[good]]=candidate[good]
        left=left[~good]
    return out


def pairwise_submatrix(D,point_ids, k):
    pairs=np.triu_indices(k,1)
    return np.column_stack([D[point_ids[:,i],point_ids[:,j]]
                            for i,j in zip(*pairs)])


def observed_cloud_feature(D, probs, rng, centers, m, L, max_k, bandwidth=1., full=False):
    # The point IDs here are independent conditional on each latent geometry.
    ids=rng.choice(len(probs),size=m,p=probs)
    features=[]
    for k in range(2,max_k+1):
        if k>m: continue
        if full:
            tuples=np.asarray(list(itertools.permutations(range(m),k)),dtype=np.int32)
        else:
            tuples=sample_distinct_rows(rng,m,L,k)
        matrix=pairwise_submatrix(D,ids[tuples],k)
        g=weighted_features(matrix,centers[k],bandwidth=bandwidth)
        weights=np.sqrt(2.**(-(k-1))*2.**(-np.arange(1,len(g)+1)))
        features.extend(g*weights)
    return np.asarray(features)


def empirical_measure_features(D, probabilities, rng, centers, L, max_k, bandwidth=1.):
    """Monte Carlo integration of measured monthly empirical *zone measure*.

    Draws are iid from the FIXED empirical probability vector. This is an
    approximation to its deterministic V-statistic (not independent monthly
    observational replicates, and not U-statistics from raw taxi trips).
    """
    features=[]
    for k in range(2,max_k+1):
        ids=rng.choice(len(probabilities),size=(L,k),p=probabilities)
        matrix=pairwise_submatrix(D,ids,k)
        g=weighted_features(matrix,centers[k],bandwidth=bandwidth)
        weights=np.sqrt(2.**(-(k-1))*2.**(-np.arange(1,len(g)+1)))
        features.extend(g*weights)
    return np.asarray(features)


# 3. CUSUM, PSD block covariance, Gaussian bridges --------------------------

def cusum(Y: np.ndarray, eta=.1):
    Y=np.asarray(Y,dtype=float);T=len(Y)
    if T<10 or Y.ndim!=2 or not np.isfinite(Y).all(): raise ValueError("bad time series")
    j=np.arange(T+1);total=Y.sum(axis=0)
    C=np.vstack((np.zeros((1,Y.shape[1])),np.cumsum(Y,axis=0)))/np.sqrt(T)
    C-=np.outer(j/T,total/np.sqrt(T))
    norm=np.linalg.norm(C,axis=1)
    start=int(math.ceil(eta*T)); stop=int(math.floor((1-eta)*T))
    mask=np.zeros(T+1,dtype=bool); mask[start:stop+1]=True
    masked=np.where(mask,norm,-np.inf)
    loc=int(np.argmax(masked))
    return {"stat":float(norm[loc]),"cp_index":loc,"cp_fraction":loc/T,
            "CUSUM":C,"norm":norm,"mask":mask}


def covariance_blocks(Y,block):
    T,d=Y.shape;B=T//block
    if B<3: raise ValueError(f"Too few complete blocks B={B} for b={block}, T={T}")
    Z=Y-Y.mean(axis=0)
    V=Z[:B*block].reshape(B,block,d).sum(axis=1)/np.sqrt(block)
    # same estimator as the theorem: 1/B sum V_j outer V_j
    Sigma=(V.T@V)/B
    return Sigma,B


def gaussian_bridge_calibration(Y, block, eta, reps, rng, batch_size=80, return_samples=False):
    # Orthogonal invariance: Gaussian bridge's norm depends only on eigenvalues
    # of the PSD covariance. Exact on the same discrete grid as sample statistic.
    Sigma,B=covariance_blocks(Y,block)
    eig=np.linalg.eigvalsh(Sigma)
    eig=np.maximum(eig,0)
    eig=eig[eig>max(1e-14,eig.max(initial=0)*1e-12)]
    T=len(Y)
    if not len(eig):
        return {"degenerate":True,"draws":np.zeros(reps),"trace":0.,"rank":0,"blocks":B}
    low=int(math.ceil(eta*T));high=int(math.floor((1-eta)*T))
    u=np.arange(1,T+1,dtype=float)/T
    maxima=[]
    for first in range(0,reps,batch_size):
        n=min(batch_size,reps-first)
        inc=rng.standard_normal((n,T,len(eig)))/math.sqrt(T)
        W=np.cumsum(inc,axis=1)
        bridge=W-u[None,:,None]*W[:,-1:, :]
        squared=np.einsum("btr,r->bt",bridge*bridge,eig,optimize=True)
        maxima.extend(np.sqrt(np.max(squared[:,low-1:high],axis=1)).tolist())
    return {"degenerate":False,"draws":np.asarray(maxima),"trace":float(eig.sum()),
            "rank":int(len(eig)),"blocks":B}


def calibrated_test(Y, rng, eta=.1, block=None, B_gauss=399, alpha=.05, return_boot=False):
    T=len(Y);b=int(block if block is not None else max(3,round(T**(1/3))))
    stat=cusum(Y,eta)
    cal=gaussian_bridge_calibration(Y,b,eta,B_gauss,rng)
    if cal["degenerate"]:
        return {"stat":stat["stat"],"cp_index":stat["cp_index"],
                "cp_fraction":stat["cp_fraction"],"pvalue":np.nan,
                "critical":np.nan,"reject":np.nan,"degenerate":True,
                "cov_trace":0.,"cov_rank":0,"block":b,"bridge":stat["norm"],
                "bootstrap":cal["draws"] if return_boot else None}
    samples=cal["draws"]
    # Plus-one Monte Carlo p-value avoids exactly zero; critical interpolated.
    p=(1+np.sum(samples>=stat["stat"]))/(len(samples)+1)
    crit=float(np.quantile(samples,1-alpha))
    return {"stat":stat["stat"],"cp_index":stat["cp_index"],
            "cp_fraction":stat["cp_fraction"],"pvalue":float(p),
            "critical":crit,"reject":bool(p<=alpha),"degenerate":False,
            "cov_trace":cal["trace"],"cov_rank":cal["rank"],
            "block":b,"bridge":stat["norm"],
            "bootstrap":samples if return_boot else None}


def wilson(k,n,z=1.96):
    if n<=0:return (np.nan,np.nan)
    p=k/n;den=1+z*z/n
    centre=(p+z*z/(2*n))/den
    half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return max(0,centre-half),min(1,centre+half)


# 4. Gaussian finite-memory random geometries ------------------------------

def simulate_geometry_sequence(T,rng,q=0,scenario="null",theta=.5,
                               noise=.35,local_strength=2.):
    # innovations across time q-dependence exactly when a fixed window is used
    eps=rng.standard_normal(T+q)
    latent=sum(eps[j:j+T] for j in range(q+1))/np.sqrt(q+1)
    u=ndtr(latent)
    cats=[]
    for t,z in enumerate(u):
        post=(t>=int(T*theta))
        if scenario=="null":
            cat="complete" if z<noise else "star"
        elif scenario=="blindspot":
            cat="complete" if z<noise else ("path" if post else "star")
        elif scenario=="support":
            cat="cycle" if z<noise else ("path" if post else "star")
        elif scenario=="measure":
            cat="complete" if z<noise else ("star_mass" if post else "star")
        elif scenario=="scale":
            cat="complete" if z<noise else ("star_scale" if post else "star")
        elif scenario=="local":
            p_path=min(.95,local_strength/np.sqrt(T)) if post else 0.
            cat="complete" if z<noise else ("path" if z<noise+(1-noise)*p_path else "star")
        else: raise ValueError(scenario)
        cats.append(cat)
    return cats


def space_for_cat(cat):
    if cat=="star_mass":return graph_metric("star",masses=[.46,.18,.18,.18])
    if cat=="star_scale":return graph_metric("star",scale=1.3)
    return graph_metric(cat)


def growth_cloud_min(T,rate):
    if rate=="slow":return max(3,int(np.ceil(np.log(np.log(T+np.exp(np.e))))))
    if rate=="moderate":return max(8,int(np.ceil(T**.25))*2)
    if rate=="ample":return max(18,int(np.ceil(T**.5)))
    raise ValueError(rate)


def generate_features(T, rng, q, scenario, max_k=3, R=6,L=300,
                      growth="moderate", theta=.5,local_strength=2.,
                      cloud_m=None,full=False):
    cats=simulate_geometry_sequence(T,rng,q,scenario,theta=theta,
                                    local_strength=local_strength)
    m_min=cloud_m if cloud_m is not None else growth_cloud_min(T,growth)
    actual_k=min(max_k,m_min)
    centers=dictionary_for_metric(actual_k,R,"graph")
    feature_rows=[]
    for t,cat in enumerate(cats):
        D,p=space_for_cat(cat)
        # strictly heterogeneous sizes, but min observation count = m_min
        m=m_min + (t%3)*2
        yy=observed_cloud_feature(D,p,rng,centers,m,L,actual_k,full=full)
        feature_rows.append(yy)
    return np.asarray(feature_rows),cats,m_min,actual_k


def monte_carlo(out,profile,seed):
    folder=out/"02_monte_carlo";folder.mkdir(parents=True,exist_ok=True)
    if profile=="smoke":
        setups=[(50,0,"null","moderate"),(50,1,"blindspot","moderate"),
                (60,0,"blindspot","slow")]
        nrep=10;Bgauss=79;L=120;orders=(2,3)
    elif profile=="pilot":
        setups=[(100,0,"null","moderate"),(150,1,"null","moderate"),
                (150,3,"null","moderate"),(200,0,"blindspot","moderate"),
                (200,1,"blindspot","moderate"),(200,1,"measure","moderate"),
                (200,1,"scale","moderate"),(200,0,"local","moderate"),
                (200,1,"null","slow"),(200,1,"null","ample")]
        nrep=100;Bgauss=299;L=400;orders=(2,3)
    else:
        setups=[]
        for T in (100,200,400,800):
            for q in (0,1,3):
                setups.append((T,q,"null","moderate"))
            for scenario in ("blindspot","measure","scale","local"):
                setups.append((T,1,scenario,"moderate"))
            setups.extend([(T,1,"null","slow"),(T,1,"null","ample")])
        nrep=2000;Bgauss=999;L=1500;orders=(2,3)
    rows=[]
    # Each replicate gets a separate PCG64 stream, with common observed features
    # for order-2 and order-3 comparisons within that replicate.
    for setting_idx,(T,q,scenario,growth) in enumerate(setups):
        tic=time.monotonic()
        for rep in range(nrep):
            # independent stream for geometry/points and calibration
            rng=np.random.default_rng(np.random.SeedSequence([seed,setting_idx,rep,101]))
            YY,cats,mmin,k=generate_features(T,rng,q,scenario,max_k=max(orders),R=6,
                                        L=L,growth=growth)
            for order in orders:
                if order>k:continue
                used_dim=(order-1)*6 # 2->6 and 3->12
                Y=YY[:,:used_dim]
                calrng=np.random.default_rng(np.random.SeedSequence([seed,setting_idx,rep,order,202]))
                start=time.monotonic()
                test=calibrated_test(Y,calrng,B_gauss=Bgauss)
                rows.append(dict(T=T,q=q,scenario=scenario,growth=growth,
                                 order=order,rep=rep,m_min=mmin,L=L,
                                 statistic=test["stat"],critical=test["critical"],
                                 pvalue=test["pvalue"],reject=test["reject"],
                                 degenerate=test["degenerate"],
                                 cp_fraction=test["cp_fraction"],
                                 cp_abs_error=(abs(test["cp_fraction"]-.5)
                                               if scenario!="null" else np.nan),
                                 block=test["block"],cov_trace=test["cov_trace"],
                                 fit_seconds=time.monotonic()-start))
        print(f"[MC] {scenario:9s} T={T:4d} q={q} growth={growth:9s} "
              f"reps={nrep} elapsed={time.monotonic()-tic:.1f}s",flush=True)
        pd.DataFrame(rows).to_csv(folder/"replicate_results_checkpoint.csv",index=False)
    df=pd.DataFrame(rows)
    write_table(df,folder,"replicate_results","All independent Monte Carlo runs and calibration outcomes")
    grp_cols=["scenario","growth","T","q","order","m_min"]
    summary=[]
    for key,part in df.groupby(grp_cols,sort=True):
        valid=part.loc[part.degenerate==False]
        n=len(valid);k=int(valid.reject.astype(bool).sum()) if n else 0
        ci=wilson(k,n)
        summary.append(dict(zip(grp_cols,key))|{"valid_reps":n,
                       "degenerate_reps":len(part)-n,
                       "rejection_rate":k/n if n else np.nan,
                       "wilson_low":ci[0],"wilson_high":ci[1],
                       "mc_se":np.sqrt((k/n)*(1-k/n)/n) if n else np.nan,
                       "median_cp_error":(valid.cp_abs_error.median() if valid.cp_abs_error.notna().any() else np.nan),
                       "mean_runtime_s":valid.fit_seconds.mean()} )
    sd=pd.DataFrame(summary)
    write_table(sd,folder,"summary_rates","Rejection frequencies with Wilson intervals and Monte Carlo SEs")
    for type_ in ("null","blindspot","measure","local"):
        sub=sd[(sd.scenario==type_)&(sd.growth=="moderate")]
        if sub.empty:continue
        fig,ax=plt.subplots(figsize=(7.5,4.6))
        for order in sorted(sub.order.unique()):
            for q in sorted(sub.q.unique()):
                xx=sub[(sub.order==order)&(sub.q==q)].sort_values("T")
                if len(xx)==0:continue
                y=xx.rejection_rate.to_numpy(float)
                ax.errorbar(xx["T"].to_numpy(int),y,
                            yerr=[y-xx.wilson_low.to_numpy(float),
                                  xx.wilson_high.to_numpy(float)-y],
                            marker="o" if order==2 else "s",capsize=3,
                            linestyle="--" if order==2 else "-",
                            color=COLORS["blue"] if order==2 else COLORS["orange"],
                            label=f"Order {order}; q={q}")
        ax.axhline(.05,color=COLORS["gray"],lw=1.2,ls=":",label="Nominal 5%")
        ax.set_ylim(bottom=0,top=min(1.,max(.16,float(sub.wilson_high.max())+.1)))
        ax.set_xlabel("Number of observed geometries T")
        ax.set_ylabel("Rejection rate (95% Wilson interval)")
        ax.set_title(f"{type_.capitalize()} study: size or power of geometric CUSUM")
        ax.legend(fontsize=8,ncol=2)
        save_fig(fig,folder,f"fig_mc_{type_}")
    ndf=df[(df.scenario=="null") & (df.growth=="moderate") & (df.order==3)]
    if len(ndf):
        fig,ax=plt.subplots(figsize=(7,4))
        ax.hist(ndf.pvalue.dropna(),bins=np.linspace(0,1,11),density=True,color=COLORS["teal"],alpha=.75)
        ax.axhline(1,color=COLORS["red"],linestyle="--",lw=1.3)
        ax.set_xlabel("Discrete Gaussian Monte Carlo p-value");ax.set_ylabel("Density")
        ax.set_title("Null calibration diagnostic (uniformity is asymptotic)")
        save_fig(fig,folder,"fig_null_pvalue_histogram")
    adf=df[(df.scenario=="blindspot") & (df.order==3)]
    if len(adf):
        fig,ax=plt.subplots(figsize=(7.5,4))
        sets=[v.cp_abs_error.dropna().values for _,v in adf.groupby("T",sort=True)]
        if sets:
            ax.boxplot(sets,positions=np.arange(len(sets)),showfliers=False)
            ax.set_xticks(np.arange(len(sets)),[str(z) for z in sorted(adf["T"].unique())])
        ax.set_ylabel("Absolute change-fraction error");ax.set_xlabel("T")
        ax.set_title("Localization error under the geometric blind-spot alternative")
        save_fig(fig,folder,"fig_cp_localization")
    return df


def computational_benchmarks(out,seed,profile):
    folder=out/"03_computation";folder.mkdir(parents=True,exist_ok=True)
    rng=np.random.default_rng(seed+333);D,p=graph_metric("star")
    mvalues=[8,12,20] if profile!="paper" else [8,12,20,36,64]
    Lvalues=[100,500,1500] if profile!="paper" else [100,500,1500,5000,20000]
    if profile=="smoke":mvalues=[8,12];Lvalues=[100,300]
    centers=dictionary_for_metric(3,6,"graph")
    rows=[]
    for m in mvalues:
        # independent large-tuple proxy within a fixed cloud: no unrealistically
        # expensive exhaustive U-statistic beyond 12 points.
        fixed_ids=rng.choice(4,size=m,p=p)
        for k in (2,3):
            ix=np.asarray(list(itertools.permutations(range(m),k)),dtype=np.int32) if m<=12 else None
            if ix is not None:
                target=weighted_features(pairwise_submatrix(D,fixed_ids[ix],k),centers[k])
            else:
                ix=sample_distinct_rows(np.random.default_rng(seed+900+m+k),m,50000,k)
                target=weighted_features(pairwise_submatrix(D,fixed_ids[ix],k),centers[k])
            for L in Lvalues:
                st=time.perf_counter()
                ix=sample_distinct_rows(rng,m,L,k)
                approx=weighted_features(pairwise_submatrix(D,fixed_ids[ix],k),centers[k])
                elapsed=time.perf_counter()-st
                rows.append(dict(m=m,k=k,L=L,
                                 runtime_seconds=elapsed,
                                 l2_error=float(np.linalg.norm(approx-target)),
                                 reference_type="exact" if m<=12 else "50k-tuple proxy"))
    dd=pd.DataFrame(rows)
    write_table(dd,folder,"incomplete_u_benchmarks","CPU sampling cost and feature approximation error")
    fig,ax=plt.subplots(figsize=(7.2,4.4))
    for (m,k),g in dd.groupby(["m","k"]):
        ax.plot(g.L,g.l2_error,marker="o",label=f"m={m}, k={k}")
    ax.set_xscale("log");ax.set_yscale("symlog",linthresh=1e-5)
    ax.set_xlabel("Random tuple evaluations per order")
    ax.set_ylabel("Feature error against full/proxy target")
    ax.set_title("Stochastic tuple approximation error (one cloud)")
    ax.legend(fontsize=8,ncol=2)
    save_fig(fig,folder,"fig_incomplete_u_error")



def robustness_study(out,seed,profile):
    """Deliberate assumption violations: exploratory diagnostics, not theorem checks."""
    folder=out/"05_robustness";folder.mkdir(parents=True,exist_ok=True)
    T=64 if profile=="smoke" else (160 if profile=="pilot" else 400)
    nrep=8 if profile=="smoke" else (80 if profile=="pilot" else 500)
    B_gauss=79 if profile=="smoke" else (299 if profile=="pilot" else 999)
    L=140 if profile=="smoke" else (400 if profile=="pilot" else 1200)
    cases=["valid_iid", "within_cloud_markov", "latent_AR1", "seasonal_nonstationary", "distance_corruption"]
    centers=dictionary_for_metric(3,6,"graph")
    result_rows=[]
    for case_idx,case in enumerate(cases):
        for rep in range(nrep):
            rng=np.random.default_rng(np.random.SeedSequence([seed,3000,case_idx,rep]))
            innovations=rng.normal(size=T+200)
            if case=="latent_AR1":
                z=np.empty(T+200);z[0]=innovations[0]
                for j in range(1,len(z)):
                    z[j]=.82*z[j-1]+np.sqrt(1-.82**2)*innovations[j]
                u=ndtr(z[-T:])
            else:
                u=ndtr(innovations[-T:])
            YY=[]
            for t,v in enumerate(u):
                chance=(.35+.22*np.sin(2*np.pi*t/12)) if case=="seasonal_nonstationary" else .35
                category="complete" if v<chance else "star"
                D,prob=space_for_cat(category)
                ids=rng.choice(4,size=12,p=prob)
                if case=="within_cloud_markov":
                    for i in range(1,len(ids)):
                        if rng.random()<.75: ids[i]=ids[i-1]
                if case=="distance_corruption":
                    noise=rng.normal(scale=.20,size=D.shape)
                    noise=(noise+noise.T)/2
                    D=np.maximum(0,D+noise)
                    np.fill_diagonal(D,0)
                    # The noisy array need not satisfy the triangle inequality.
                ff=[]
                for k in (2,3):
                    ix=sample_distinct_rows(rng,len(ids),L,k)
                    dm=pairwise_submatrix(D,ids[ix],k)
                    g=weighted_features(dm,centers[k])
                    ff.extend(g*np.sqrt(2.**(-(k-1))*2.**(-np.arange(1,7))))
                YY.append(ff)
            cal=calibrated_test(np.asarray(YY),
                    np.random.default_rng(np.random.SeedSequence([seed,4000,case_idx,rep])),
                    B_gauss=B_gauss)
            result_rows.append({"case":case,"rep":rep,"T":T,
                                "pvalue":cal["pvalue"],"reject":cal["reject"],
                                "statistic":cal["stat"],"cov_rank":cal["cov_rank"]})
        print("[robustness]",case,"reps",nrep,flush=True)
    raw=pd.DataFrame(result_rows)
    write_table(raw,folder,"robustness_replicates",
               "Out-of-assumption Monte Carlo calibration diagnostics")
    summary=[]
    for case,g in raw.groupby("case"):
        k=int(g.reject.sum());n=len(g);lo,hi=wilson(k,n)
        summary.append({"case":case,"reps":n,"rejection_rate":k/n,
                        "wilson_95_low":lo,"wilson_95_high":hi,
                        "interpretation":"exploratory; assumptions not satisfied" if case!="valid_iid" else "within finite-q assumptions"})
    sd=pd.DataFrame(summary)
    write_table(sd,folder,"robustness_summary","Assumption violations compared with an independent null")
    fig,ax=plt.subplots(figsize=(9,4.1))
    vals=sd.set_index("case").reindex(cases)
    xx=np.arange(len(vals));yy=vals.rejection_rate.to_numpy()
    ax.bar(xx,yy,color=[COLORS["teal"]]+[COLORS["orange"]]*4)
    ax.errorbar(xx,yy,yerr=[yy-vals.wilson_95_low.to_numpy(),
                              vals.wilson_95_high.to_numpy()-yy],
                fmt="none",color=COLORS["ink"],capsize=3)
    ax.axhline(.05,ls="--",color=COLORS["red"],label="Nominal reference 5%")
    ax.set_xticks(xx,[x.replace("_","\n") for x in cases]);ax.set_ylim(0,1)
    ax.set_ylabel("Rejection frequency (Wilson 95% intervals)")
    ax.set_title("Diagnostic sensitivity outside the finite-memory/i.i.d.-cloud model")
    ax.legend();save_fig(fig,folder,"fig_robustness_rejection_rates")


# 5. Official TLC downloads, schema auditing and finite metric -------------
URL_ROOT="https://d37ci6vzurychx.cloudfront.net"
LOOKUP_URL=URL_ROOT+"/misc/taxi_zone_lookup.csv"
ZONES_URL=URL_ROOT+"/misc/taxi_zones.zip"


def fetch_cached(url: str, dest: Path, attempts=3):
    dest.parent.mkdir(parents=True,exist_ok=True)
    if dest.exists() and dest.stat().st_size>500: return dest
    last=None
    for attempt in range(attempts):
        try:
            req=urllib.request.Request(url,headers={"User-Agent":"Academic-Research-Notebook/1.0"})
            partial=dest.with_suffix(dest.suffix+".part")
            sha=hashlib.sha256()
            with urllib.request.urlopen(req,timeout=80) as r, open(partial,"wb") as o:
                while True:
                    chunk=r.read(1024*1024)
                    if not chunk: break
                    o.write(chunk);sha.update(chunk)
            if partial.stat().st_size<500:raise IOError("implausibly short download")
            os.replace(partial,dest)
            dest.with_suffix(dest.suffix+".sha256").write_text(sha.hexdigest(),encoding="ascii")
            return dest
        except Exception as exc:
            last=exc
            print("DOWNLOAD ERROR",url,type(exc).__name__,str(exc)[:180],"attempt",attempt+1,flush=True)
            time.sleep(2*(attempt+1))
    raise RuntimeError(f"Cannot download {url}: {last}")


def get_zone_geometries(cache):
    """Returns lookup, (zone ids, fixed distances / 35 km), and polygon coords.

    Polygon representative points are transformed EPSG:2263 -> EPSG:4326
    and fixed WGS84 geodesic distances are used; 35 km is global/fixed.
    """
    import shapefile  # pyshp
    from shapely.geometry import shape as shapely_shape
    from pyproj import Transformer, Geod
    lookup=pd.read_csv(fetch_cached(LOOKUP_URL,cache/"taxi_zone_lookup.csv"))
    for needed in ["LocationID","Borough","Zone"]:
        if needed not in lookup.columns:raise ValueError(f"missing lookup column {needed}")
    if not (cache/"taxi_zones"/"taxi_zones.shp").exists():
        z=fetch_cached(ZONES_URL,cache/"taxi_zones.zip")
        with zipfile.ZipFile(z) as zf:
            # Only standard expected shapefile components, no paths traversal
            dest=cache/"taxi_zones";dest.mkdir(exist_ok=True)
            for member in zf.namelist():
                if Path(member).name in {"taxi_zones.shp","taxi_zones.shx","taxi_zones.dbf","taxi_zones.prj","taxi_zones.cpg"}:
                    (dest/Path(member).name).write_bytes(zf.read(member))
    reader=shapefile.Reader(str(cache/"taxi_zones"/"taxi_zones.shp"))
    keys=[x[0] for x in reader.fields[1:]]
    assert "LocationID" in keys, f"shapefile fields: {keys}"
    rep={};poly={}
    transformer=Transformer.from_crs("EPSG:2263","EPSG:4326",always_xy=True)
    for sr in reader.iterShapeRecords():
        zone=int(sr.record[keys.index("LocationID")])
        gg=shapely_shape(sr.shape.__geo_interface__)
        point=gg.representative_point()  # guaranteed inside a polygon
        lng,lat=transformer.transform(point.x,point.y)
        rep[zone]=(lng,lat)
        # matplotlib polygon exterior only; metadata is still retained
        polys=gg.geoms if gg.geom_type=="MultiPolygon" else [gg]
        poly[zone]=[np.asarray(p.exterior.coords) for p in polys if not p.is_empty]
    ids=sorted(set(lookup.LocationID.astype(int)) & set(rep))
    if len(ids)<200:raise ValueError("Too few valid zones")
    coords=np.asarray([rep[x] for x in ids])
    geod=Geod(ellps="WGS84")
    lon1=coords[:,0,None];lon2=coords[:,0][None,:]
    lat1=coords[:,1,None];lat2=coords[:,1][None,:]
    _,_,meters=geod.inv(np.broadcast_to(lon1,(len(ids),len(ids))),
                        np.broadcast_to(lat1,(len(ids),len(ids))),
                        np.broadcast_to(lon2,(len(ids),len(ids))),
                        np.broadcast_to(lat2,(len(ids),len(ids))))
    D=meters/35000.
    np.fill_diagonal(D,0.)
    if not np.allclose(D,D.T,rtol=0,atol=1e-9):raise ValueError("non symmetric geodesic matrix")
    # A metric on points from a geodesic surface is triangle inequality;
    # detect duplicates: distinct zones with identical representative points.
    if (D+np.eye(len(ids))*10).min()<1e-12:raise ValueError("duplicate representative points")
    return lookup,ids,D,poly


def extract_tlc_month(year,month,ids,cache,retain_raw=False):
    import pyarrow.parquet as pq
    path=cache/"trip_data"/f"yellow_tripdata_{year}-{month:02d}.parquet"
    fetch_cached(f"{URL_ROOT}/trip-data/{path.name}",path)
    pf=pq.ParquetFile(path)
    schema=pf.schema_arrow.names
    date_candidates=["tpep_pickup_datetime","pickup_datetime"]
    loc_candidates=["PULocationID","pulocationid","pickup_location_id"]
    date_col=next((v for v in date_candidates if v in schema),None)
    loc_col=next((v for v in loc_candidates if v in schema),None)
    if date_col is None or loc_col is None:raise ValueError(f"Unknown parquet schema: {schema}")
    counts=np.zeros(len(ids),dtype=np.int64)
    id_to_i={z:i for i,z in enumerate(ids)}
    raw=0; good_date=0; good_zone=0; retained=0
    for batch in pf.iter_batches(batch_size=150_000,columns=[date_col,loc_col]):
        frame=batch.to_pandas()
        raw+=len(frame)
        dt=pd.to_datetime(frame[date_col],errors="coerce")
        valid_date=(dt.dt.year==year)&(dt.dt.month==month)
        loc=pd.to_numeric(frame[loc_col],errors="coerce")
        valid_zone=loc.isin(ids)
        good_date+=int(valid_date.sum());good_zone+=int(valid_zone.sum())
        selected=loc[valid_date & valid_zone].astype(int)
        vc=selected.value_counts()
        for z,c in vc.items():counts[id_to_i[int(z)]]+=int(c)
        retained+=len(selected)
    audit={"year":year,"month":month,"raw_rows":raw,"date_matching_rows":good_date,
           "recognized_zone_rows":good_zone,"valid_rows":retained,
           "excluded_rows":raw-retained,"valid_fraction":retained/max(1,raw),
           "file_bytes":path.stat().st_size,"parquet_sha256":(path.with_suffix(path.suffix+".sha256").read_text().strip()
                               if path.with_suffix(path.suffix+".sha256").exists()
                               else "checksum_not_available_for_preexisting_cache"),
           "source_url":f"{URL_ROOT}/trip-data/{path.name}",
           "schema_pickup_time":date_col,"schema_pickup_zone":loc_col}
    if retained<1000:raise RuntimeError(f"Insufficient valid trips {year}-{month:02d}: {retained}")
    return counts,audit


def months_inclusive(start,end):
    y,m=map(int,start.split("-"));ey,em=map(int,end.split("-"))
    if (y,m)>(ey,em):raise ValueError("End before start")
    while (y,m)<=(ey,em):
        yield y,m
        m+=1
        if m==13:y,m=y+1,1


def draw_zone_choropleth(ax,polys,ids,vals,title,cmap="coolwarm",vcenter=0):
    vals=np.asarray(vals,float)
    good=np.isfinite(vals)
    maxabs=max(.0001,float(np.nanmax(abs(vals[good])))) if good.any() else 1.
    norm=TwoSlopeNorm(vmin=-maxabs,vcenter=0,vmax=maxabs) if vcenter==0 else Normalize(vmin=vals.min(),vmax=vals.max())
    palette=plt.get_cmap(cmap)
    parts=[];colors=[]
    for zone,value in zip(ids,vals):
        for poly in polys.get(zone,[]):
            parts.append(poly)
            colors.append(palette(norm(value)) if np.isfinite(value) else (0.8,0.8,0.8,1))
    collection=PolyCollection(parts,facecolors=colors,edgecolors="#BBBBBB",linewidth=.12)
    ax.add_collection(collection);ax.autoscale_view();ax.set_aspect("equal");ax.axis("off")
    ax.set_title(title)
    return plt.cm.ScalarMappable(norm=norm,cmap=palette)


def real_tlc_application(out,cache,profile,seed,strict=False,start=None,end=None,keep_raw=False):
    folder=out/"04_nyc_tlc";folder.mkdir(parents=True,exist_ok=True)
    if (start is None) != (end is None):
        raise ValueError("Provide BOTH --start and --end, or neither")
    if start is None or end is None:
        start,end = ("2023-01","2023-12") if profile=="smoke" else \
                    (("2019-01","2021-12") if profile=="pilot" else ("2018-01","2024-12"))
    lookup,ids,D,polys=get_zone_geometries(cache)
    meta={"source":"NYC TLC official monthly yellow taxi trip records",
          "source_page":"https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page",
          "zone_metric":"WGS84 geodesic between fixed interior zone representative points / 35km",
          "eligible_zone_count":len(ids),"start":start,"end":end,
          "monthly_parquet_origin":"https://d37ci6vzurychx.cloudfront.net/trip-data/",
          "zone_shapefile_origin":ZONES_URL,
          "zone_lookup_origin":LOOKUP_URL,
          "inferential_status":"exploratory; month-level strict stationarity and finite q-dependence unverified"}
    write_json(folder/"metadata.json",meta)
    dates=list(months_inclusive(start,end))
    month_cache=cache/"monthly_counts";month_cache.mkdir(exist_ok=True,parents=True)
    counts=[];audits=[]
    for i,(year,month) in enumerate(dates):
        f=month_cache/f"yellow_counts_{year}-{month:02d}.npz"
        af=month_cache/f"yellow_audit_{year}-{month:02d}.json"
        if f.exists() and af.exists():
            dat=np.load(f);stored_ids=dat["zone_ids"]
            if np.array_equal(stored_ids,ids):
                cc=dat["counts"]
                with open(af) as fh:audit=json.load(fh)
            else: f.unlink();af.unlink();cc,audit=extract_tlc_month(year,month,ids,cache)
        else:
            cc,audit=extract_tlc_month(year,month,ids,cache)
        np.savez_compressed(f,zone_ids=np.array(ids),counts=cc)
        write_json(af,audit)
        if not keep_raw:
            (cache/"trip_data"/f"yellow_tripdata_{year}-{month:02d}.parquet").unlink(missing_ok=True)
        counts.append(cc);audits.append(audit)
        print(f"[TLC] {year}-{month:02d} ({i+1}/{len(dates)}), validated trips: {cc.sum():,}",flush=True)
    counts=np.stack(counts).astype(np.int64)
    data_audit=pd.DataFrame(audits)
    write_table(data_audit,folder,"monthly_quality_audit","Official parquet record counts, exclusions and sources")
    np.savez_compressed(folder/"monthly_zone_counts.npz",zone_ids=np.asarray(ids),
                        months=np.asarray([f"{y}-{m:02d}" for y,m in dates]),counts=counts)
    probabilities=counts/counts.sum(axis=1,keepdims=True)
    safe=np.maximum(probabilities,np.finfo(float).tiny)
    entropy=-(probabilities*np.log(safe)).sum(axis=1)
    squared_conc=(probabilities**2).sum(axis=1)
    labels=[f"{y}-{m:02d}" for y,m in dates]
    detail=pd.DataFrame({"month":labels,"trips_retained":counts.sum(axis=1),
                         "entropy_nats":entropy,"effective_zones":np.exp(entropy),
                         "herfindahl":squared_conc})
    write_table(detail,folder,"monthly_spatial_descriptives","Spatial entropy and concentration of TLC pickups")
    fig,ax=plt.subplots(figsize=(11,4))
    xx=np.arange(len(labels));ax.plot(xx,counts.sum(axis=1)/1e6,color=COLORS["blue"])
    ax.set_ylabel("Validated monthly trips (millions)")
    ax.set_xlabel("Calendar month");ax.set_title("Monthly valid yellow-taxi pickup records")
    ticks=np.arange(0,len(labels),max(1,len(labels)//12))
    ax.set_xticks(ticks,[labels[i] for i in ticks],rotation=45,ha="right")
    save_fig(fig,folder,"fig_tlc_monthly_counts")
    fig,ax=plt.subplots(figsize=(11,4))
    ax.plot(xx,np.exp(entropy),color=COLORS["teal"],lw=2)
    ax.set_xlabel("Calendar month");ax.set_ylabel("Effective number of pickup zones, exp(entropy)")
    ax.set_title("Diversity of monthly pickup-zone distributions")
    ax.set_xticks(ticks,[labels[i] for i in ticks],rotation=45,ha="right")
    save_fig(fig,folder,"fig_tlc_zone_entropy")
    # Monte Carlo integration of each observed monthly empirical measure.
    # Repeat independent tuple streams to audit numerical approximation error.
    L = 1200 if profile=="smoke" else (7000 if profile=="pilot" else 30000)
    dictionary=dictionary_for_metric(3,6,"nyc")
    Y=[];Y_alt=[]
    for t,p in enumerate(probabilities):
        rg=np.random.default_rng(np.random.SeedSequence([seed,700,t]))
        rg2=np.random.default_rng(np.random.SeedSequence([seed,701,t]))
        Y.append(empirical_measure_features(D,p,rg,dictionary,L,3))
        Y_alt.append(empirical_measure_features(D,p,rg2,dictionary,L,3))
    Y=np.asarray(Y);Y_alt=np.asarray(Y_alt)
    approximation_error=float(np.linalg.norm(Y-Y_alt,axis=1).mean()/math.sqrt(2))
    np.savez_compressed(folder/"monthly_geometric_features.npz",Y=Y,Y_repeat=Y_alt,
                        dictionary_order2=dictionary[2],dictionary_order3=dictionary[3],
                        months=np.asarray(labels),L=L)
    temporal_sd = float(np.sqrt(np.sum(np.var(Y,axis=0,ddof=1))))
    approximation_ratio=approximation_error/max(temporal_sd,1e-12)
    provenance={"tuple_evaluations_per_month_per_order":L,
                "temporal_feature_variation_rms":temporal_sd,
                "integration_error_to_temporal_variation_ratio":approximation_ratio,
                "integration_design":"with replacement from empirical zone frequencies",
                "mean_replication_feature_discrepancy_div_sqrt2":approximation_error,
                "warning":"Integration Monte Carlo variance is not a substitute for monthly time-series variation"}
    write_json(folder/"integration_diagnostics.json",provenance)
    # Descriptive seasonality diagnostics, not a proof of the q-dependent null.
    if len(labels)>=24:
        fig,ax=plt.subplots(figsize=(10,4.3))
        table_seas=detail.copy()
        table_seas["year"]=pd.to_datetime(table_seas.month).dt.year
        table_seas["month_num"]=pd.to_datetime(table_seas.month).dt.month
        pivot=table_seas.pivot(index="year",columns="month_num",values="effective_zones")
        im=ax.imshow(pivot.to_numpy(),aspect="auto",interpolation="nearest",cmap="viridis")
        ax.set_yticks(np.arange(len(pivot)),pivot.index.astype(str))
        ax.set_xticks(np.arange(12),["J","F","M","A","M","J","J","A","S","O","N","D"])
        ax.set_xlabel("Calendar month");ax.set_ylabel("Year")
        ax.set_title("Year-by-month spatial diversity: descriptive seasonality diagnostic")
        fig.colorbar(im,ax=ax,label="Effective pickup zones")
        save_fig(fig,folder,"fig_tlc_entropy_seasonality_heatmap")
    centred = Y - Y.mean(axis=0)
    acf_rows=[]
    for lag in range(1,min(13,len(Y)//3)):
        denom=np.sum(centred*centred)
        acf=float(np.sum(centred[:-lag]*centred[lag:])/denom) if denom>0 else np.nan
        acf_rows.append({"lag_months":lag,"normalized_Hilbert_inner_product":acf})
    acf_df=pd.DataFrame(acf_rows)
    write_table(acf_df,folder,"hilbert_acf_diagnostics",
                "Empirical normalized feature autocorrelation (diagnostic only)")
    fig,ax=plt.subplots(figsize=(7,3.7))
    ax.axhline(0,color=COLORS["gray"],lw=1)
    ax.vlines(acf_df.lag_months,0,acf_df.normalized_Hilbert_inner_product,color=COLORS["blue"],lw=2)
    ax.scatter(acf_df.lag_months,acf_df.normalized_Hilbert_inner_product,color=COLORS["blue"])
    ax.set_xlabel("Lag in months");ax.set_ylabel("Normalized Hilbert autocorrelation")
    ax.set_title("Dependence diagnostic for the monthly geometric signatures")
    save_fig(fig,folder,"fig_tlc_hilbert_autocorrelation")
    # Full sample and pre-specified order-2 comparator; 1 shared metric and feature dictionary.
    rows=[];response={}
    for order in (2,3):
        dim=6*(order-1)
        for exponent in (.25,1/3,.40):
            b=max(3,int(round(len(Y)**exponent)))
            if len(Y)//b<3: continue
            rng=np.random.default_rng(np.random.SeedSequence([seed,1400,order,int(exponent*100)]))
            result=calibrated_test(Y[:,:dim],rng,eta=.1,block=b,
                                   B_gauss=(299 if profile=="smoke" else 1999),return_boot=True)
            rows.append({"order":order,"block":b,"exponent":exponent,
                         "T":len(Y),"statistic":result["stat"],
                         "gaussian_p_exploratory":result["pvalue"],
                         "critical":result["critical"],"estimated_change_index":result["cp_index"],
                         "first_post_change_month":(labels[result["cp_index"]]
                           if result["cp_index"]<len(labels) else "after-series"),
                         "covariance_rank":result["cov_rank"],
                         "covariance_trace":result["cov_trace"],
                         "calibration_degenerate":result["degenerate"]})
            if order==3 and abs(exponent-1/3)<1e-9:response=result
    write_table(pd.DataFrame(rows),folder,"change_point_sensitivity",
                "Exploratory Gaussian plug-in calibration (theorem assumptions unverified for taxi series)")
    if not response:raise RuntimeError("primary order-three calibration missing")
    fig,ax=plt.subplots(figsize=(11,4))
    u=np.arange(len(Y)+1);ax.plot(u,response["bridge"],color=COLORS["blue"],lw=2)
    ax.axhline(response["critical"],color=COLORS["orange"],ls="--",
               label="Gaussian simulated null supremum critical value")
    ax.axvline(response["cp_index"],color=COLORS["red"],ls=":",
               label=f"Maximum at index {response['cp_index']}")
    ax.set_xlabel("Month index");ax.set_ylabel("Hilbert-space CUSUM norm")
    ax.set_title("Monthly spatial geometry change-point profile (exploratory)")
    ax.legend(loc="best");save_fig(fig,folder,"fig_tlc_cusum_profile")
    fig,ax=plt.subplots(figsize=(7,4.5))
    ax.hist(response["bootstrap"],bins=30,color=COLORS["teal"],alpha=.78)
    ax.axvline(response["stat"],ls="--",lw=2,color=COLORS["red"],label="Observed maximum")
    ax.axvline(response["critical"],ls=":",lw=2,color=COLORS["ink"],label="95% simulated quantile")
    ax.set_xlabel("Gaussian bridge supremum");ax.set_ylabel("Number of draws")
    ax.set_title("Positive-semidefinite block Gaussian calibration")
    ax.legend();save_fig(fig,folder,"fig_tlc_bootstrap_distribution")
    cp=response["cp_index"]
    if cp<3 or cp>=len(Y)-3: raise RuntimeError("estimated split too close to sample boundary")
    pre=probabilities[:cp].mean(axis=0)
    post=probabilities[cp:].mean(axis=0)
    changes=post-pre
    zones=lookup.set_index("LocationID")
    table=pd.DataFrame({"LocationID":ids,
                        "Borough":[zones.loc[z,"Borough"] for z in ids],
                        "Zone":[zones.loc[z,"Zone"] for z in ids],
                        "pre_probability":pre,"post_probability":post,
                        "change_percentage_points":100*changes})
    write_table(table.sort_values("change_percentage_points",ascending=False),folder,
                "zone_probability_changes","Average monthly normalized zone probabilities before and after estimated break")
    fig,ax=plt.subplots(figsize=(10,7.2))
    sm=draw_zone_choropleth(ax,polys,ids,100*changes,
           "Change in average monthly pickup share (percentage points)")
    fig.colorbar(sm,ax=ax,shrink=.72,label="After minus before (percentage points)")
    save_fig(fig,folder,"fig_tlc_pickup_zone_change_map")
    # Month-by-month borough shares; an observational descriptor, not a test.
    boroughs=np.asarray([str(zones.loc[z,"Borough"]) for z in ids])
    names=[s for s in sorted(set(boroughs)) if s not in ("Unknown","N/A")]
    fig,ax=plt.subplots(figsize=(11,4.3))
    for b in names:
        share=probabilities[:,boroughs==b].sum(axis=1)
        ax.plot(xx,share,lw=1.5,label=b)
    ax.set_xlabel("Month");ax.set_ylabel("Monthly probability mass")
    ax.set_title("Borough-level pickup composition (descriptive)")
    ax.set_xticks(ticks,[labels[i] for i in ticks],rotation=45,ha="right")
    ax.legend(ncol=3,fontsize=8)
    save_fig(fig,folder,"fig_tlc_borough_composition")
    write_json(folder/"interpretation_notes.json",{
        "status":"EXPLORATORY MODEL-BASED INFERENCE; NOT VALIDATED THEOREM-LEVEL SIZE",
        "stationarity":"Seasonal patterns, COVID, data reporting shifts, and persistent dynamics violate/obscure null",
        "covariance":"Nonoverlapping block Gaussian plug-in, finite T, sensitivity to b included",
        "month_measure":"Normalized empirical frequency vector of eligible TLC pickup zones",
        "bootstrap":"Simulated Gaussian bridge using PSD long-run covariance estimate, not resampled taxi trips",
        "resampling":"Tuple sampling approximates an empirical V-statistic; it never creates fresh monthly observations",
        "estimated_split_semantics":"CUSUM index j splits first j months from remaining T-j; mapped month is the first AFTER the split",
        "data_scope":f"{start} to {end}","rows_quality_checked":int(counts.sum())})
    return response


# 6. Verification and entry point ------------------------------------------

def unit_tests():
    rng=np.random.default_rng(444)
    D,p=graph_metric("star")
    assert np.allclose(D,D.T) and np.max(np.abs(np.diag(D)))==0
    assert exact_pattern_law("star",2)==exact_pattern_law("path",2)
    c=dictionary_for_metric(3,6,"graph")
    a=observed_cloud_feature(D,p,rng,c,m=7,L=3000,max_k=3)
    assert a.shape==(12,) and np.isfinite(a).all()
    # Conditioned on a finite observed cloud, incomplete U-stat expectation
    # converges to complete; testing second-order alone to keep work bounded.
    cloud=np.array([0,1,2,3,1,0]);
    idx=np.asarray(list(itertools.permutations(range(len(cloud)),2)),dtype=np.int32)
    target=weighted_features(pairwise_submatrix(D,cloud[idx],2),c[2])
    idx=sample_distinct_rows(np.random.default_rng(8),len(cloud),30000,2)
    est=weighted_features(pairwise_submatrix(D,cloud[idx],2),c[2])
    assert np.max(np.abs(est-target))<.015,(est,target)
    YY=np.random.default_rng(7).normal(size=(60,12))
    b=5;Sig,B=covariance_blocks(YY,b)
    assert np.linalg.eigvalsh(Sig).min()>-1e-12
    cal=calibrated_test(YY,np.random.default_rng(8),B_gauss=60)
    assert 0<=cal["pvalue"]<=1
    # Global translation of D does not make sense for metric but simultaneous
    # label permutation must not change empirical measure feature expectations.
    perm=np.array([3,1,0,2]); Dp=D[np.ix_(perm,perm)];pp=p[perm]
    assert np.allclose(empirical_measure_features(D,p,np.random.default_rng(2),c,500,3),
             empirical_measure_features(Dp,pp,np.random.default_rng(2),c,500,3),atol=.12)
    # Monte Carlo p-value criticals do not have spurious exactly-zero p-values.
    return True


def main():
    parser=argparse.ArgumentParser()
    parser.add_argument("--profile",choices=["smoke","pilot","paper"],default="pilot")
    parser.add_argument("--mode",choices=["all","simulation","real"],default="all")
    parser.add_argument("--seed",type=int,default=20261024)
    parser.add_argument("--output",default="",help="Absolute output root directory")
    parser.add_argument("--cache",default="",help="Persistent download/cache directory")
    parser.add_argument("--start",default="",help="YYYY-MM override TLC start")
    parser.add_argument("--end",default="",help="YYYY-MM override TLC end")
    parser.add_argument("--keep-raw",action="store_true",help="Retain each monthly downloaded Parquet (many GB)")
    parser.add_argument("--strict-real",action="store_true",help="Raise if remote TLC data fail")
    args=parser.parse_args()
    visual_style()
    now=datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    out=Path(args.output).resolve() if args.output else ROOT/"runs"/f"{args.profile}_{now}"
    cache=Path(args.cache).resolve() if args.cache else ROOT/"tlc_cache"
    out.mkdir(parents=True,exist_ok=True)
    shutil.copy2(Path(__file__),out/"annealed_study_SOURCE.py")
    status={"profile":args.profile,"mode":args.mode,"seed":args.seed,
            "created_utc":datetime.now(timezone.utc).isoformat(),
            "python":platform.python_version(),"platform":platform.platform(),
            "versions":{},"completed_sections":[],"failures":[],
            "run_type":"smoke/pilot results are PRELIMINARY, not publication grade" if args.profile!="paper" else "paper configuration"}
    for pkg in ["numpy","pandas","scipy","matplotlib","pyarrow","shapely","pyproj","pyshp"]:
        try:status["versions"][pkg]=importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError: status["versions"][pkg]="not installed"
    write_json(out/"RUN_MANIFEST.json",status)
    try:
        assert unit_tests()
        status["completed_sections"].append("unit tests")
        exact_verification(out)
        status["completed_sections"].append("exact arithmetic")
        if args.mode in ("simulation","all"):
            monte_carlo(out,args.profile,args.seed)
            status["completed_sections"].append("Monte Carlo simulation")
            computational_benchmarks(out,args.seed,args.profile)
            status["completed_sections"].append("computational benchmarks")
            robustness_study(out,args.seed,args.profile)
            status["completed_sections"].append("misspecification diagnostics")
        if args.mode in ("real","all"):
            try:
                real_tlc_application(out,cache,args.profile,args.seed,
                                     start=args.start or None,end=args.end or None,
                                     keep_raw=args.keep_raw)
                status["completed_sections"].append("official TLC real data")
            except Exception as e:
                status["failures"].append({"section":"real data","exception":repr(e),
                                          "traceback":traceback.format_exc()})
                (out/"REAL_DATA_FAILURE.txt").write_text(traceback.format_exc(),encoding="utf8")
                print("TLC real-data section FAILED, simulations remain available:",repr(e),flush=True)
                if args.strict_real:raise
        status["success"]=(len(status["failures"])==0)
    except BaseException as e:
        status["failures"].append({"section":"pipeline","exception":repr(e),
                                  "traceback":traceback.format_exc()})
        status["success"]=False
        raise
    finally:
        write_json(out/"RUN_MANIFEST.json",status)
        (out/"OUTPUT_CONTENTS.txt").write_text("\n".join(str(p.relative_to(out)) for p in sorted(out.rglob("*")) if p.is_file()),encoding="utf8")
        zip_path=archive_results(out)
        if os.environ.get("COLAB_RELEASE_DOWNLOAD")=="1":
            try:
                from google.colab import files
                files.download(str(zip_path))
            except Exception as exc:print("Download from file sidebar instead:",exc)
    print("Completed:",status["completed_sections"],"failures",len(status["failures"]))

if __name__=="__main__":main()


## 4 · Execute the study
Tables appear as cells are computed; every figure is exported to both PNG and vector PDF. Downloads of original TLC records are streamed by Parquet batches and resumed through cached monthly zone counts on subsequent runs. Running the full paper configuration is intentionally computationally intensive.

In [ ]:
import sys, subprocess, shlex
from pathlib import Path
args = [sys.executable, "annealed_study.py", "--profile", PROFILE, "--mode", MODE,
        "--seed", str(MASTER_SEED)]
if START_MONTH:
    args += ["--start", START_MONTH, "--end", END_MONTH]
if KEEP_PARQUET:
    args += ["--keep-raw"]
print("Running:", " ".join(shlex.quote(x) for x in args), flush=True)
subprocess.run(args, check=True)

## 5 · Inspect outputs inline
Below: tables and selected figures generated by this actual run, never placeholder results. The paper-profile results should be reviewed for all diagnostics and sensitivity analyses before writing inferential claims.

In [ ]:
from IPython.display import display, Image, Markdown
import pandas as pd
run_folders = sorted(Path("runs").glob(f"{PROFILE}_*"), key=lambda x:x.stat().st_mtime)
assert run_folders, "No completed run folder found"
LATEST_RUN = run_folders[-1]
print("Run folder:", LATEST_RUN)
for filename in ["01_exact/moment_matching_exact.csv",
                 "02_monte_carlo/summary_rates.csv",
                 "03_computation/incomplete_u_benchmarks.csv",
                 "05_robustness/robustness_summary.csv",
                 "04_nyc_tlc/change_point_sensitivity.csv"]:
    path=LATEST_RUN/filename
    if path.exists():
        display(Markdown("### " + filename))
        display(pd.read_csv(path).head(20))
plots = sorted(LATEST_RUN.rglob("fig_*.png"))
print(f"Found {len(plots)} figure files; displaying at most 16")
for path in plots[:16]:
    display(Markdown("**" + str(path.relative_to(LATEST_RUN)) + "**"))
    display(Image(filename=str(path),width=850))

## 6 · Download one ZIP of **all** output tables, plots, data audits and the executed code
A ZIP is automatically produced by the engine even if the real-data download fails. Check `RUN_MANIFEST.json` and, if present, `REAL_DATA_FAILURE.txt`. The ZIP intentionally does not embed many gigabytes of original TLC Parquet files.

In [ ]:
from pathlib import Path
zips = sorted(Path("runs").glob(f"annealed_gromov_outputs_{LATEST_RUN.name}.zip"))
assert len(zips)==1, "Expected exactly one ZIP for the run"
OUTPUT_ZIP = zips[0]
print(f"ARCHIVE: {OUTPUT_ZIP.resolve()} ({OUTPUT_ZIP.stat().st_size/1e6:.2f} MB)")
try:
    from google.colab import files
    files.download(str(OUTPUT_ZIP))
except ImportError:
    display(Markdown(f"[Local ZIP archive]({OUTPUT_ZIP})"))

## 7 · Methodological cautions for journal submission
A finite order-2/3 dictionary is a computational approximation, not proof of an omnibus finite-sample test. The star/path null must have a nondegenerate auxiliary random geometry; deterministic geometries do not test the nondegenerate calibration theorem. TLC monthly counts yield empirical measures, not independent repeated geometries; its calibrated p-values are exploratory until the dependence/seasonality problem is independently resolved. The source and exact configuration accompany every ZIP.